# STEP 6 - Rendering-verified rerun of the core results (XLM-R)

Local HarfBuzz checks on 12 Sep found three problems with Steps 3-5:

1. The `semi` flood is **not invisible**: 43-88% of attacked words render identically,
   and some show dotted circles. The 99.6% / 49% numbers used a partly visible attack.
2. The position-aware defense is the exact inverse of that attack, so "100% recovery"
   holds by construction.
3. An **adaptive** attacker can put ZWNJ/ZWJ right before hasant in ya-phala. That is
   invisible, and position-aware keeps it.

This notebook re-measures everything under one rule: **an insertion is allowed only if
HarfBuzz renders the word identically in all 8 reference fonts.** It also checks a
confound: flooding inflates tokens, so at `max_length=128` toxic words can be truncated.

**Settings:** GPU **T4 x2** (one is used) | Internet **ON** | Add data **`naurosromim/bdshs`**

**Run it with `Save Version -> Save & Run All (Commit)`**, not interactively. It takes about
2 hours, and a commit keeps running if you close the browser. When it finishes, open the
version -> **Output** -> download all files, then send me the `STEP 6 SUMMARY` block
from the log.


In [ ]:
# 1. Install
!pip install -q "emoji==1.7.0" regex ftfy uharfbuzz
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
%%writefile /kaggle/working/render_attack.py
"""
Rendering-verified zero-width attacks and defenses for Bangla.

Threat model
------------
An insertion counts as imperceptible only if HarfBuzz shapes the word to the
same glyph ids and glyph positions, in every reference font, with and without
it. HarfBuzz is the shaper used by Chrome, Android and Firefox.

The earlier rule-based attacks (strict / semi) are kept for comparison, but
every number in the paper should come from the render-gated functions here.
"""
import os
import re
import random
import urllib.request
import unicodedata
from functools import lru_cache

import uharfbuzz as hb

ZWNJ, ZWJ, ZWSP = "‌", "‍", "​"
HASANT = "্"
ZW = (ZWNJ, ZWJ, ZWSP)

# Open-licensed Bangla fonts, all downloadable on Kaggle.
FONT_URLS = {
    "NotoSansBengali":  "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSansBengali/unhinted/ttf/NotoSansBengali-Regular.ttf",
    "NotoSerifBengali": "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSerifBengali/unhinted/ttf/NotoSerifBengali-Regular.ttf",
    "HindSiliguri":     "https://raw.githubusercontent.com/google/fonts/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
    "TiroBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/tirobangla/TiroBangla-Regular.ttf",
    "BalooDa2":         "https://raw.githubusercontent.com/google/fonts/main/ofl/balooda2/BalooDa2%5Bwght%5D.ttf",
    "AnekBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/anekbangla/AnekBangla%5Bwdth,wght%5D.ttf",
    "Galada":           "https://raw.githubusercontent.com/google/fonts/main/ofl/galada/Galada-Regular.ttf",
    "Mina":             "https://raw.githubusercontent.com/google/fonts/main/ofl/mina/Mina-Regular.ttf",
}


def fetch_fonts(dest, names=None):
    os.makedirs(dest, exist_ok=True)
    paths = {}
    for name in names or FONT_URLS:
        p = os.path.join(dest, name + ".ttf")
        if not os.path.exists(p):
            urllib.request.urlretrieve(FONT_URLS[name], p)
        paths[name] = p
    return paths


def is_bangla(ch):
    """True for one Bangla code point.

    An empty string returns False, so callers may pass a missing neighbour
    (start or end of string) without guarding first."""
    return len(ch) == 1 and 0x0980 <= ord(ch) <= 0x09FF


# ------------------------------------------------------------------ slots

def inert_slots(word):
    """Between two Bangla chars, neither of them hasant (the old `semi` set)."""
    return [i for i in range(1, len(word))
            if is_bangla(word[i - 1]) and is_bangla(word[i])
            and HASANT not in (word[i - 1], word[i])]


def strict_slots(word):
    """Both neighbours Bangla base letters (the old `strict` set)."""
    return [i for i in inert_slots(word)
            if unicodedata.category(word[i - 1]) == "Lo"
            and unicodedata.category(word[i]) == "Lo"]


def hasant_slots(word):
    """Immediately before or after a hasant. Position-aware defense keeps these."""
    out = set()
    for h, c in enumerate(word):
        if c != HASANT:
            continue
        if h > 0 and is_bangla(word[h - 1]):
            out.add(h)
        if h + 1 < len(word) and is_bangla(word[h + 1]):
            out.add(h + 1)
    return sorted(out)


SLOT_MODES = {
    "inert":  inert_slots,
    "strict": strict_slots,
    "hasant": hasant_slots,
    "both":   lambda w: inert_slots(w) + hasant_slots(w),
}


def _build(word, chosen, char):
    return "".join((char if i in chosen else "") + c for i, c in enumerate(word))


def _parts(text):
    return re.split(r"(\s+)", text)


# --------------------------------------------------------------- renderer

class Renderer:
    def __init__(self, font_paths):
        self.fonts = {n: hb.Font(hb.Face(hb.Blob.from_file_path(p), 0))
                      for n, p in font_paths.items()}
        self.names = list(self.fonts)
        self._shape = lru_cache(maxsize=2_000_000)(self._shape_uncached)

    def _shape_uncached(self, name, text):
        buf = hb.Buffer()
        buf.add_str(text)
        buf.guess_segment_properties()
        buf.flags = hb.BufferFlags.REMOVE_DEFAULT_IGNORABLES
        hb.shape(self.fonts[name], buf, {})
        return (tuple(g.codepoint for g in buf.glyph_infos),
                tuple((p.x_advance, p.y_advance, p.x_offset, p.y_offset)
                      for p in buf.glyph_positions))

    def same(self, a, b, fonts=None):
        return all(self._shape(n, a) == self._shape(n, b) for n in (fonts or self.names))

    def visible_in(self, a, b):
        """Names of fonts in which a and b render differently."""
        return [n for n in self.names if self._shape(n, a) != self._shape(n, b)]

    # ------------------------------------------------------------ attacks

    def attack(self, text, budget="ALL", char=ZWNJ, seed=0, mode="inert", fonts=None):
        """Insert zero-width chars only where the word still renders identically.

        Candidates are shuffled and accepted one at a time, each checked against
        the clean word with all previously accepted insertions in place.
        Returns (attacked_text, n_inserted)."""
        parts = _parts(text)
        rng = random.Random(seed)
        slots = SLOT_MODES[mode]
        cands = [(pi, i) for pi, p in enumerate(parts)
                 if p and not p.isspace() for i in slots(p)]
        rng.shuffle(cands)
        chosen, n = {}, 0
        for pi, i in cands:
            if budget != "ALL" and n >= budget:
                break
            trial = chosen.get(pi, set()) | {i}
            if self.same(parts[pi], _build(parts[pi], trial, char), fonts):
                chosen[pi] = trial
                n += 1
        out = "".join(_build(p, chosen[pi], char) if pi in chosen else p
                      for pi, p in enumerate(parts))
        return out, n

    # ------------------------------------------------------------ defense

    def defend_render_equivalence(self, text, fonts=None):
        """Delete a zero-width char iff removing it leaves rendering unchanged in
        every reference font. Only chars whose nearest non-zero-width neighbours
        on both sides are Bangla are considered, so emoji ZWJ sequences survive."""
        parts = _parts(text)
        for pi, w in enumerate(parts):
            if not any(z in w for z in ZW):
                continue
            i = 0
            while i < len(w):
                if w[i] in ZW:
                    l = i - 1
                    while l >= 0 and w[l] in ZW:
                        l -= 1
                    r = i + 1
                    while r < len(w) and w[r] in ZW:
                        r += 1
                    if l >= 0 and r < len(w) and is_bangla(w[l]) and is_bangla(w[r]):
                        cand = w[:i] + w[i + 1:]
                        if self.same(w, cand, fonts):
                            w = cand
                            continue
                i += 1
            parts[pi] = w
        return "".join(parts)


# ------------------------------------------------ rule-based (for comparison)

def attack_ungated(text, budget="ALL", char=ZWNJ, seed=0, mode="inert"):
    parts = _parts(text)
    rng = random.Random(seed)
    slots = SLOT_MODES[mode]
    cands = [(pi, i) for pi, p in enumerate(parts) if p and not p.isspace() for i in slots(p)]
    rng.shuffle(cands)
    if budget != "ALL":
        cands = cands[:budget]
    chosen = {}
    for pi, i in cands:
        chosen.setdefault(pi, set()).add(i)
    out = "".join(_build(p, chosen[pi], char) if pi in chosen else p for pi, p in enumerate(parts))
    return out, len(cands)


def defend_strip_all(text):
    for z in ZW:
        text = text.replace(z, "")
    return text


def defend_nfc(text):
    return unicodedata.normalize("NFC", text)


def defend_position_aware(text):
    out = []
    for i, ch in enumerate(text):
        if ch in (ZWNJ, ZWJ):
            prev = text[i - 1] if i > 0 else ""
            nxt = text[i + 1] if i + 1 < len(text) else ""
            if HASANT in (prev, nxt):
                out.append(ch)
            continue
        if ch == ZWSP:
            continue
        out.append(ch)
    return "".join(out)


def zw_stats(texts):
    docs = total = functional = 0
    for t in texts:
        hit = False
        for i, ch in enumerate(t):
            if ch not in (ZWNJ, ZWJ):
                continue
            hit = True
            total += 1
            prev = t[i - 1] if i > 0 else ""
            nxt = t[i + 1] if i + 1 < len(t) else ""
            functional += HASANT in (prev, nxt)
        docs += hit
    return dict(n_docs=len(texts), docs_with_zw=docs, total_zw=total,
                functional=functional, inert=total - functional)


In [ ]:
# 3. Imports, fonts, self-test
import sys, os, glob, time, random, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/kaggle/working")
import importlib, render_attack as ra; importlib.reload(ra)
import numpy as np, pandas as pd, torch
from collections import Counter
from functools import lru_cache
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
from normalizer import normalize

DEV, OUT = "cuda", "/kaggle/working"
FONT_PATHS = ra.fetch_fonts(f"{OUT}/fonts")
R = ra.Renderer(FONT_PATHS)
print("GPU:", torch.cuda.get_device_name(0), "| fonts:", R.names)

_s = "আমাদের দেশ বাংলাদেশ ধন্যবাদ স্যার"
_a, _n = R.attack(_s, "ALL", mode="both")
assert _n > 0 and R.same(_s, _a) and R.defend_render_equivalence(_a) == _s
print("self-test ok: inserted", _n, "invisible chars, render-equivalence restored the string")

In [ ]:
# 4. Data - FULL training set this time
paths = glob.glob("/kaggle/input/**/*.csv", recursive=True)
pick = lambda k: [p for p in paths if k in os.path.basename(p).lower()][0]
raw_train, raw_test = pd.read_csv(pick("train")), pd.read_csv(pick("test"))
obj = [c for c in raw_train.columns if raw_train[c].dtype == object]
TEXT = max(obj, key=lambda c: raw_train[c].astype(str).str.len().mean())
LABEL = [c for c in raw_train.columns if c != TEXT and raw_train[c].dropna().nunique() == 2][0]

def prep(d):
    d = d[[TEXT, LABEL]].dropna().copy(); d.columns = ["text", "label"]
    d["text"] = d["text"].astype(str); d["label"] = d["label"].astype(int)
    return d[d["text"].str.strip().str.len() > 0].reset_index(drop=True)

train, test = prep(raw_train), prep(raw_test)
HATE = 1
print(f"TEXT={TEXT} LABEL={LABEL} | train {len(train)} | test {len(test)} | test hate {int((test.label == HATE).sum())}")

In [ ]:
# 5. Build every attacked set ONCE (model-independent), and measure visibility
N_ATK = 1500
rng = random.Random(0)
hate_idx = test.index[test.label == HATE].tolist()
ben_idx = test.index[test.label != HATE].tolist()
hate_txt = [test.text[i] for i in sorted(rng.sample(hate_idx, min(N_ATK, len(hate_idx))))]
ben_txt = [test.text[i] for i in sorted(rng.sample(ben_idx, min(N_ATK, len(ben_idx))))]

CONFIGS = [
    ("gated_inert_b1",            lambda t, s: R.attack(t, 1, mode="inert", seed=s)),
    ("gated_inert_b3",            lambda t, s: R.attack(t, 3, mode="inert", seed=s)),
    ("gated_inert_b5",            lambda t, s: R.attack(t, 5, mode="inert", seed=s)),
    ("gated_inert_b10",           lambda t, s: R.attack(t, 10, mode="inert", seed=s)),
    ("gated_inert_ALL",           lambda t, s: R.attack(t, "ALL", mode="inert", seed=s)),
    ("gated_inert_ALL_zwj",       lambda t, s: R.attack(t, "ALL", char=ra.ZWJ, mode="inert", seed=s)),
    ("gated_adaptive_hasant_ALL", lambda t, s: R.attack(t, "ALL", mode="hasant", seed=s)),
    ("gated_both_ALL",            lambda t, s: R.attack(t, "ALL", mode="both", seed=s)),
    ("ungated_semi_ALL_old",      lambda t, s: ra.attack_ungated(t, "ALL", mode="inert", seed=s)),
    ("gated_inert_ALL_notoOnly",  lambda t, s: R.attack(t, "ALL", mode="inert", seed=s, fonts=["NotoSansBengali"])),
]
ATK, META = {}, []
for name, fn in CONFIGS:
    t0 = time.time()
    res = [fn(t, i) for i, t in enumerate(hate_txt)]
    ATK[name] = [a for a, _ in res]
    vis = [R.visible_in(c, a) for c, (a, _) in zip(hate_txt, res)]
    META.append({"config": name,
                 "mean_inserted": round(float(np.mean([n for _, n in res])), 2),
                 "pct_with_insertion": round(float(np.mean([n > 0 for _, n in res])), 4),
                 "pct_visible_any_font": round(float(np.mean([len(v) > 0 for v in vis])), 4),
                 "secs": round(time.time() - t0, 1)})
    print(META[-1])
meta_df = pd.DataFrame(META); meta_df.to_csv(f"{OUT}/step6_attack_meta.csv", index=False)

cf = Counter(f for c, a in zip(hate_txt, ATK["gated_inert_ALL_notoOnly"]) for f in R.visible_in(c, a))
crossfont = pd.DataFrame([{"held_out_font": f, "visibly_broken": cf.get(f, 0), "of": len(hate_txt)} for f in R.names])
crossfont.to_csv(f"{OUT}/step6_crossfont.csv", index=False)
BEN_ATK = [R.attack(t, "ALL", mode="both", seed=i)[0] for i, t in enumerate(ben_txt)]

In [ ]:
# 6. Defenses: precompute outputs, exact recovery, latency, in-the-wild collateral
DEFENSES = {
    "none": lambda t: t,
    "nfc": ra.defend_nfc,
    "strip_all": ra.defend_strip_all,
    "position_aware": ra.defend_position_aware,
    "render_equivalence": R.defend_render_equivalence,
}
DEF_ATTACKS = ["gated_inert_ALL", "gated_adaptive_hasant_ALL", "gated_both_ALL"]
DEFENDED = {(a, d): [fn(t) for t in ATK[a]] for a in DEF_ATTACKS for d, fn in DEFENSES.items()}
CLEAN_DEF = {d: [fn(t) for t in test.text] for d, fn in DEFENSES.items()}

recovery = pd.DataFrame([{"attack": a, "defense": d,
    "exact_string_recovery": round(float(np.mean([x == c for x, c in zip(DEFENDED[(a, d)], hate_txt)])), 4)}
    for a in DEF_ATTACKS for d in DEFENSES])
recovery.to_csv(f"{OUT}/step6_exact_recovery.csv", index=False)

sample = ATK["gated_both_ALL"][:500]
lat = []
for d, fn in DEFENSES.items():
    if d == "render_equivalence":
        fn = ra.Renderer(FONT_PATHS).defend_render_equivalence   # cold cache = honest latency
    t0 = time.perf_counter()
    for t in sample: fn(t)
    lat.append({"defense": d, "ms_per_comment": round((time.perf_counter() - t0) / len(sample) * 1000, 4)})
latency = pd.DataFrame(lat); latency.to_csv(f"{OUT}/step6_latency.csv", index=False)

alltxt = train.text.tolist() + test.text.tolist()
wild = ra.zw_stats(alltxt)
zw_docs = [t for t in alltxt if any(z in t for z in ra.ZW)]
collateral = pd.DataFrame([{"defense": d, "docs_with_zw": len(zw_docs),
    "codepoints_changed": sum(DEFENSES[d](t) != t for t in zw_docs),
    "rendering_changed_any_font": sum(not R.same(t, DEFENSES[d](t)) for t in zw_docs)}
    for d in ["nfc", "strip_all", "position_aware", "render_equivalence"]])
collateral.to_csv(f"{OUT}/step6_wild_collateral.csv", index=False)
print(wild); print(collateral.to_string(index=False)); print(latency.to_string(index=False))

In [ ]:
# 7. Train + evaluate one (model, seed)
MAXLEN_TRAIN = 128

@lru_cache(maxsize=None)
def norm(t):
    return normalize(t)

def train_model(name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tk = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2).to(DEV)
    e = tk([norm(t) for t in train.text], truncation=True, padding="max_length",
           max_length=MAXLEN_TRAIN, return_tensors="pt")
    y = torch.tensor(train.label.values)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(TensorDataset(e["input_ids"], e["attention_mask"], y), batch_size=32, shuffle=True, generator=g)
    EPOCHS = 2
    opt = torch.optim.AdamW(mdl.parameters(), lr=2e-5)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-5, total_steps=EPOCHS * len(dl), pct_start=0.1)
    sc = torch.cuda.amp.GradScaler(); mdl.train()
    for ep in range(EPOCHS):
        for st, (i_, m_, l_) in enumerate(dl):
            i_, m_, l_ = i_.to(DEV), m_.to(DEV), l_.to(DEV); opt.zero_grad()
            with torch.cuda.amp.autocast():
                loss = mdl(input_ids=i_, attention_mask=m_, labels=l_).loss
            sc.scale(loss).backward(); sc.step(opt); sc.update(); sch.step()
            if st % 400 == 0:
                print(f"  {name} seed{seed} ep{ep} {st}/{len(dl)} loss {loss.item():.4f}")
    mdl.eval()
    return tk, mdl

@torch.no_grad()
def predict(tk, mdl, texts, max_len=512, bs=128):
    ids = tk([norm(t) for t in texts], truncation=True, max_length=max_len)["input_ids"]
    lens = np.array([len(x) for x in ids])
    order = np.argsort(lens)
    preds = np.empty(len(texts), dtype=int)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        b = tk.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt")
        with torch.cuda.amp.autocast():
            lg = mdl(input_ids=b["input_ids"].to(DEV), attention_mask=b["attention_mask"].to(DEV)).logits
        preds[idx] = lg.float().argmax(-1).cpu().numpy()
    return preds, lens

def evaluate(name, seed):
    tk, mdl = train_model(name, seed)
    tokcls = type(tk).__name__
    ty = test.label.values
    clean, ladder, defense = [], [], []
    for d in DEFENSES:
        p, _ = predict(tk, mdl, CLEAN_DEF[d])
        clean.append({"model": name, "seed": seed, "defense_on_clean": d,
                      "acc": round(accuracy_score(ty, p), 4), "f1": round(f1_score(ty, p), 4)})
    hp, _ = predict(tk, mdl, hate_txt)
    tgt = np.where(hp == HATE)[0]
    for cfg, _ in CONFIGS:
        if cfg == "gated_inert_ALL_notoOnly":
            continue
        a = [ATK[cfg][i] for i in tgt]
        p512, lens = predict(tk, mdl, a)
        p128, _ = predict(tk, mdl, a, max_len=128)
        ladder.append({"model": name, "seed": seed, "tokenizer": tokcls, "config": cfg, "n_targets": len(tgt),
                       "evasion": round(float((p512 != HATE).mean()), 4),
                       "evasion_at_maxlen128": round(float((p128 != HATE).mean()), 4),
                       "pct_over_128_tokens": round(float((lens > 128).mean()), 4)})
    for a_name in DEF_ATTACKS:
        for d in DEFENSES:
            p, _ = predict(tk, mdl, [DEFENDED[(a_name, d)][i] for i in tgt])
            defense.append({"model": name, "seed": seed, "attack": a_name, "defense": d,
                            "evasion": round(float((p != HATE).mean()), 4)})
    bp, _ = predict(tk, mdl, ben_txt)
    bpa, _ = predict(tk, mdl, BEN_ATK)
    benign = {"model": name, "seed": seed,
              "benign_fpr_clean": round(float((bp == HATE).mean()), 4),
              "benign_fpr_attacked": round(float((bpa == HATE).mean()), 4),
              "benign_flip_rate": round(float((bp != bpa).mean()), 4)}
    del mdl; torch.cuda.empty_cache()
    return clean, ladder, defense, benign

In [ ]:
# 8. Run: XLM-R x 3 seeds + BanglaBERT control. Results are saved after every run.
RUNS = [("xlm-roberta-base", 42), ("xlm-roberta-base", 43), ("xlm-roberta-base", 44), ("csebuetnlp/banglabert", 42)]
CL, LA, DE, BE = [], [], [], []
for name, seed in RUNS:
    t0 = time.time()
    try:
        c, l, d, b = evaluate(name, seed)
    except Exception as ex:
        print("FAILED", name, seed, type(ex).__name__, ex)
        continue
    CL += c; LA += l; DE += d; BE.append(b)
    pd.DataFrame(CL).to_csv(f"{OUT}/step6_clean_cost.csv", index=False)
    pd.DataFrame(LA).to_csv(f"{OUT}/step6_ladder.csv", index=False)
    pd.DataFrame(DE).to_csv(f"{OUT}/step6_defense.csv", index=False)
    pd.DataFrame(BE).to_csv(f"{OUT}/step6_benign.csv", index=False)
    print(f"== {name} seed {seed} done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 9. SUMMARY  <-- copy this whole block back
def agg(rows, keys, cols):
    df = pd.DataFrame(rows)
    g = df.groupby(keys, sort=False)[cols].agg(["mean", "std"]).round(4)
    g.columns = [f"{c}_{s}" for c, s in g.columns]
    return g.reset_index()

pd.set_option("display.width", 250)
print("=" * 90); print("STEP 6 SUMMARY"); print("=" * 90)
print(f"train {len(train)} | test {len(test)} | attacked hate comments {len(hate_txt)} | fonts {len(R.names)}")
print("\n-- attack construction (visibility checked in all fonts) --"); print(meta_df.to_string(index=False))
print("\n-- cross-font: attack gated on Noto Sans only, visibly broken in --"); print(crossfont.to_string(index=False))
print("\n-- evasion ladder (mean/std over seeds) --")
print(agg(LA, ["model", "config"], ["n_targets", "evasion", "evasion_at_maxlen128", "pct_over_128_tokens"]).to_string(index=False))
print("\n-- evasion after defense --"); print(agg(DE, ["model", "attack", "defense"], ["evasion"]).to_string(index=False))
print("\n-- exact string recovery (by construction, not a model result) --"); print(recovery.to_string(index=False))
print("\n-- clean accuracy cost of each defense --"); print(agg(CL, ["model", "defense_on_clean"], ["acc", "f1"]).to_string(index=False))
print("\n-- benign control (attack applied to non-hate comments) --"); print(pd.DataFrame(BE).to_string(index=False))
print("\n-- in-the-wild --"); print(wild); print(collateral.to_string(index=False))
print("\n-- defense latency --"); print(latency.to_string(index=False))
print("=" * 90)